# Search-03b (C#) — Bases de données de motifs (Pattern Databases) pour le 15-puzzle

> **Partie 3 — Recherche avancée.** Version C# / .NET du notebook [Search-03b-PatternDatabases](Search-03b-PatternDatabases.ipynb) (Python). Ce notebook fait partie de l'Epic de parité .NET ⇄ Python [#4956](https://github.com/jsboige/CoursIA/issues/4956) : chaque concept porté invoque le **vrai moteur** — ici, l'algorithme est porté *from scratch* en C# (les PDB n'ont pas de solveur off-the-shelf, c'est l'archétype du précalcul heuristique).

## 1. Le problème, et pourquoi Manhattan ne suffit plus

Le **15-puzzle** (taquin 4×4) est le banc d'essai historique de la recherche
heuristique : 15 tuiles + 1 case vide, on glisse les tuiles vers la configuration
ordonnée. Résoudre *à l'optimum* est **NP-difficile** et constitue, depuis
**Korf (1985)**, la référence pour mesurer un solveur de recherche.

[Search-3](../Part1-Foundations/Search-03-Informed.ipynb) a introduit **IDA\***
et l'heuristique **Manhattan** (somme des distances individuelles des tuiles).
Manhattan est *admissible* (ne surestime jamais), donc IDA\* + Manhattan est
optimal. Mais elle ignore les **interactions** entre tuiles (deux tuiles qui se
bloquent coûtent plus que la somme de leurs distances) : sur les instances
difficiles de Korf (~50 coups optimaux), IDA\* + Manhattan explore des
**centaines de milliards** de nœuds.

La SOTA, depuis **Culberson & Schaeffer (1996)** puis **Korf & Felner (2002)**,
consiste à *précalculer* une heuristique admissible bien plus fine : la **base de
données de motifs**. Ce notebook construit d'abord une PDB simple (concept), puis
la **PDB additive** qui est, quarante ans après Korf, la référence absolue du
15-puzzle.

## 2. Formalisation du 15-puzzle

État = tableau de 16 entiers (cellules en row-major), `0` = case vide. But = tuiles
1..15 en ordre, vide en bas à droite.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

int[] GOAL = {1,2,3,4, 5,6,7,8, 9,10,11,12, 13,14,15,0};
const int N = 4;

int[] Neighbors(int cell) {
    int r = cell / N, c = cell % N;
    var res = new List<int>();
    if (r > 0) res.Add(cell - N);
    if (r < N - 1) res.Add(cell + N);
    if (c > 0) res.Add(cell - 1);
    if (c < N - 1) res.Add(cell + 1);
    return res.ToArray();
}

int[] Move(int[] state, int blank, int nxt) {
    var s = (int[])state.Clone();
    (s[blank], s[nxt]) = (s[nxt], s[blank]);
    return s;
}

int[][] Successors(int[] state) {
    int b = Array.IndexOf(state, 0);
    return Neighbors(b).Select(n => Move(state, b, n)).ToArray();
}

Console.WriteLine($"OK : le but a {Successors(GOAL).Length} successeurs");

OK : le but a 2 successeurs


## 3. Heuristique de Manhattan (référence, vu en Search-3)

In [2]:
Dictionary<int,int> GOAL_POS = GOAL.Select((t,i) => (t,i)).ToDictionary(p => p.t, p => p.i);

int Manhattan(int[] state) {
    int h = 0;
    for (int cell=0; cell<state.Length; cell++) {
        int t = state[cell];
        if (t == 0) continue;
        int gr = GOAL_POS[t] / N, gc = GOAL_POS[t] % N;
        int cr = cell / N, cc = cell % N;
        h += Math.Abs(gr - cr) + Math.Abs(gc - cc);
    }
    return h;
}

Console.WriteLine($"Manhattan(but) = {Manhattan(GOAL)}");

Manhattan(but) = 0


### 3.b Mélange reproductible (marche aléatoire)

On brouille le but par une marche aléatoire longue (90-130 pas), ce qui produit
des instances *réalistement difficiles* (optimal typiquement ~38-46 coups) — bien
plus coûteuses à résoudre que les instances jouets où Manhattan est instantané.

In [3]:
int[] Scramble(int[] goal, int nSteps, int seed) {
    var rng = new Random(seed);
    int[] s = (int[])goal.Clone();
    int[] prev = null;
    for (int i=0; i<nSteps; i++) {
        int b = Array.IndexOf(s, 0);
        var cands = Neighbors(b).Where(n => !Move(s,b,n).SequenceEqual(prev ?? Array.Empty<int>())).ToList();
        int nxt = cands[rng.Next(cands.Count)];
        prev = (int[])s.Clone();
        s = Move(s, b, nxt);
    }
    return s;
}

int[] INST = Scramble(GOAL, 100, seed: 11);
Console.WriteLine($"Instance de démo : ({string.Join(", ", INST)})");
Console.WriteLine($"Manhattan = {Manhattan(INST)} (borne inférieure ; l'optimal est nettement supérieur)");

Instance de démo : (1, 12, 0, 3, 9, 2, 11, 4, 10, 13, 15, 7, 5, 8, 6, 14)


Manhattan = 26 (borne inférieure ; l'optimal est nettement supérieur)


## 4. Concept : base de données de motifs

L'idée de Culberson & Schaeffer (1996) :

1. **Abstraction.** On choisit un *motif* (sous-ensemble de tuiles). Les autres
   deviennent des jokers indifférenciés (*wildcards*).
2. **Parcours rétrograde.** Dans l'espace abstrait, BFS *depuis le but abstrait*,
   enregistrant pour chaque état abstrait le **coût optimal** pour y ramener le motif.
3. **Consultation.** Pour un état concret, on l'abstrait et on lit le coût en O(1).

**Admissibilité.** L'espace abstrait *relâche* le problème (les jokers se
substituent gratuitement) → le coût abstrait minore le coût réel → heuristique
admissible. IDA\* + PDB reste donc optimal.

**Fonction de rang.** Pour stocker la table dans un tableau compact, on *range*
chaque état abstrait vers un entier : rang de permutation des positions du motif
`P(16,k)` (code de Lehmer), multiplié par le nombre de cellules restantes pour le vide.

In [4]:
int P(int n, int k) { int r = 1; for (int i=0; i<k; i++) r *= (n - i); return r; }

int PermRank(int[] positions, int basis=16) {
    var avail = new List<int>(); for (int i=0; i<basis; i++) avail.Add(i);
    int rank = 0; int k = positions.Length;
    for (int i=0; i<k; i++) {
        int idx = avail.IndexOf(positions[i]);
        rank += idx * P(basis - 1 - i, k - 1 - i);
        avail.RemoveAt(idx);
    }
    return rank;
}

int RankAbstract(int[] pos, int blank, int k) {
    int rPerm = PermRank(pos, 16);
    var remaining = new List<int>(); for (int c=0; c<16; c++) if (!pos.Contains(c)) remaining.Add(c);
    return rPerm * (16 - k) + remaining.IndexOf(blank);
}

// Test d'injectivité : deux états abstraits *distincts* ne doivent jamais
// partager le même rang (l'abstraction est many-to-one par construction, on teste
// donc l'injectivité du rang, pas l'absence de collisions de classes).
var rngTest = new Random(1);
var seen = new Dictionary<int,(int[] pos,int blank)>();
int collisions = 0;
for (int i=0; i<20000; i++) {
    var pool = Enumerable.Range(0,16).OrderBy(x => rngTest.Next()).Take(6).ToList();
    var pos = pool.Take(5).ToArray(); int blank = pool.Last();
    int rk = RankAbstract(pos, blank, 5);
    if (seen.TryGetValue(rk, out var prev)) { if (!prev.pos.SequenceEqual(pos) || prev.blank != blank) collisions++; }
    else seen[rk] = (pos, blank);
}
Console.WriteLine($"Injectivité du rang : {seen.Count} états abstraits testés, {collisions} collision (attendu 0)");

Injectivité du rang : 19975 états abstraits testés, 0 collision (attendu 0)


## 5. PDB simple : construction par BFS rétrograde

On construit d'abord une PDB sur **un** motif de tuiles. BFS = coût unitaire =
distances optimales depuis le but abstrait. Cette PDB unique couvre un sous-ensemble
de tuiles ; elle servira de brique, puis la **PDB additive** (section 6) en combinera
plusieurs pour couvrir les 15 tuiles.

In [5]:
(byte[] table, int nSeen) BuildPdb(int[] group) {
    int k = group.Length;
    int[] goalPos = group.Select(t => Array.IndexOf(GOAL, t)).ToArray();
    int goalBlank = Array.IndexOf(GOAL, 0);
    int size = P(16, k) * (16 - k);
    var table = new byte[size];
    for (int i=0; i<size; i++) table[i] = 255;
    table[RankAbstract(goalPos, goalBlank, k)] = 0;
    var q = new Queue<(int[] pos, int blank)>();
    q.Enqueue((goalPos, goalBlank));
    int nSeen = 1;
    while (q.Count > 0) {
        var (pos, blank) = q.Dequeue();
        int d = table[RankAbstract(pos, blank, k)];
        var pset = new HashSet<int>(pos);
        foreach (int n in Neighbors(blank)) {
            int[] newPos; int newBlank;
            if (pset.Contains(n)) { int ti = Array.IndexOf(pos, n); newPos = (int[])pos.Clone(); newPos[ti] = blank; newBlank = n; }
            else { newPos = pos; newBlank = n; }
            int rk = RankAbstract(newPos, newBlank, k);
            if (table[rk] == 255) { table[rk] = (byte)(d + 1); q.Enqueue((newPos, newBlank)); nSeen++; }
        }
    }
    return (table, nSeen);
}

int PdbLookup(int[] state, byte[] table, int[] group) {
    int[] pos = group.Select(t => Array.IndexOf(state, t)).ToArray();
    return table[RankAbstract(pos, Array.IndexOf(state, 0), group.Length)];
}

int[] PATTERN = {1, 2, 3, 4};
var sw = System.Diagnostics.Stopwatch.StartNew();
var (PDB5, n5) = BuildPdb(PATTERN);
sw.Stop();
int dmax = PDB5.Where(v => v < 255).Max();
Console.WriteLine($"PDB simple ({PATTERN.Length} tuiles) : {n5} états en {sw.Elapsed.TotalSeconds:F1}s, distance max = {dmax}");

PDB simple (4 tuiles) : 524160 états en 4,5s, distance max = 50


### Lecture de la construction : 524 160 = P(16,4) × 12, et la table EST la solution précalculée

La sortie `524 160 états en 3,9 s, distance max = 50` se décompose entièrement à la main. Le BFS rétrograde énumère les positions possibles des 4 tuiles du motif **et** de la case vide dans les 16 cases : 16 × 15 × 14 × 13 = 43 680 arrangements de tuiles, × 12 positions restantes pour la vide = **524 160** états. Le `distance max = 50` dit ensuite que le pire sous-puzzle (les 4 tuiles les plus mal placées) exige 50 coups à lui seul : la table encode donc des optima *partiels* déjà très coûteux, que l'heuristique ira lire en O(1) au lieu de les recalculer. C'est tout le compromis : ~3,9 s de construction une fois, des millions de consultations gratuites ensuite.

Un point qui surprend : le BFS part du BUT et remonte les coups **à l'envers**. Chaque état atteint à profondeur d est donc à distance *exactement* d de l'objectif — la table est un oracle de distance exacte sur le sous-espace, pas une estimation.

## 6. La SOTA : PDB *additives* (Korf & Felner 2002)

Une PDB unique sur quelques tuiles n'informe qu'une fraction du puzzle. L'idée de
génie de Korf & Felner (2002) : **partitionner** les 15 tuiles en groupes *disjoints*
et construire une PDB par groupe, en ne comptant — dans chaque BFS — **que les
mouvements des tuiles du groupe**. Les coûts deviennent alors **additifs** :

$$h(s) = \sum_{\text{groupes } G} \mathrm{PDB}_G(s)$$

est une heuristique **admissible** : chaque coup du taquin déplace exactement une
tuile, donc la somme des minima par groupe disjoint minore le nombre total de coups.

On utilise une partition **4-4-4-3** (les groupes couvrent les 15 tuiles). Le BFS
devient un **0-1 BFS** (deque) : déplacer une tuile du groupe coûte 1, déplacer la
case vide à travers un joker coûte 0.

In [6]:
int[][] GROUPS = { new[]{1,2,3,4}, new[]{5,6,7,8}, new[]{9,10,11,12}, new[]{13,14,15} };

(byte[] table, int nSeen) BuildAdditivePdb(int[] group) {
    int k = group.Length;
    int[] goalPos = group.Select(t => Array.IndexOf(GOAL, t)).ToArray();
    int goalBlank = Array.IndexOf(GOAL, 0);
    int size = P(16, k) * (16 - k);
    var dist = new byte[size];
    for (int i=0; i<size; i++) dist[i] = 255;
    dist[RankAbstract(goalPos, goalBlank, k)] = 0;
    var dq = new LinkedList<(int[] pos, int blank)>();
    dq.AddLast((goalPos, goalBlank));
    int nSeen = 1;
    while (dq.Count > 0) {
        var (pos, blank) = dq.First.Value; dq.RemoveFirst();
        int d = dist[RankAbstract(pos, blank, k)];
        var pset = new HashSet<int>(pos);
        foreach (int n in Neighbors(blank)) {
            int[] newPos; int newBlank; int w;
            if (pset.Contains(n)) { int ti = Array.IndexOf(pos, n); newPos = (int[])pos.Clone(); newPos[ti] = blank; newBlank = n; w = 1; }
            else { newPos = pos; newBlank = n; w = 0; }
            int rk = RankAbstract(newPos, newBlank, k);
            if (dist[rk] == 255) {
                dist[rk] = (byte)(d + w);
                if (w == 0) dq.AddFirst((newPos, newBlank)); else dq.AddLast((newPos, newBlank));
                nSeen++;
            }
        }
    }
    return (dist, nSeen);
}

sw.Restart();
var ADDITIVE = GROUPS.Select(g => BuildAdditivePdb(g)).ToArray();
byte[][] ADD_TBLS = ADDITIVE.Select(a => a.table).ToArray();
sw.Stop();
Console.WriteLine($"PDB additives (4-4-4-3) construites en {sw.Elapsed.TotalSeconds:F1}s");
foreach (var (g, a) in GROUPS.Zip(ADDITIVE)) {
    int dm = a.table.Where(v => v < 255).Max();
    Console.WriteLine($"  groupe ({string.Join(",", g)}): {a.nSeen} états, dist max = {dm}");
}

PDB additives (4-4-4-3) construites en 11,1s


  groupe (1,2,3,4): 524160 états, dist max = 21


  groupe (5,6,7,8): 524160 états, dist max = 17


  groupe (9,10,11,12): 524160 états, dist max = 19


  groupe (13,14,15): 43680 états, dist max = 16


### Lecture des groupes : pourquoi la somme des 4 tables reste admissible

La sortie donne quatre tables : trois de 524 160 états (groupes de 4 tuiles) et une de 43 680 (groupe de 3 : P(16,3) = 16 × 15 × 14 = 3 360 arrangements, × 13 positions restantes pour la case vide = 43 680 — le vide est encodé comme pour les groupes de 4). Les distances max se lisent 21, 17, 19 et 16. Le point structurel : chaque coup du 15-puzzle déplace **une seule tuile**, qui appartient à **un seul** groupe — les coûts des groupes sont disjoints par coup, donc leur somme ne peut jamais surestimer la distance réelle. C'est la différence radicale avec la PDB classique de Culberson & Schaeffer : on y prenait le **max** de plusieurs vues du même puzzle ; Korf & Felner partitionnent en groupes **disjoints** pour pouvoir **sommer**. La même somme appliquée à des groupes qui se chevaucheraient casserait l'admissibilité immédiatement (un même coup compté deux fois).

## 7. IDA\* instrumenté

Rappel de [Search-3](../Part1-Foundations/Search-03-Informed.ipynb) : seuils
itératifs sur `f = g + h`, recherche en profondeur. On compte les **nœuds
explorés** (métrique du coût de recherche). On l'introduit avant l'heuristique
additive car il sert à sa vérification d'admissibilité ci-après.

On se fixe un **budget de 2 millions de nœuds** par résolution : c'est largement
assez pour résoudre les instances faciles, et cela rend visible (et rapide) la
frontière où une heuristique termine et l'autre non. *L'état est encodé en un
`long` (16 cellules × 4 bits) pour le hachage du chemin (cycle-avoidance).*

In [7]:
long Encode(int[] state) { long k = 0; foreach (int t in state) k = k * 16 + t; return k; }

(int? cost, long nodes) IdaStar(int[] start, Func<int[],int> heuristic, int[] goal=null, long nodeLimit=2_000_000) {
    goal ??= GOAL;
    int threshold = heuristic(start); long nodes = 0;
    while (true) {
        var path = new HashSet<long>(); path.Add(Encode(start));
        var (res, t, n) = Dfs(start, 0, threshold, heuristic, goal, path, nodeLimit - nodes);
        nodes += n;
        if (res.HasValue) return (res, nodes);
        if (t == int.MaxValue || nodes >= nodeLimit) return (null, nodes);
        threshold = t;
    }
}

(int? cost, int nextThreshold, long nodes) Dfs(int[] state, int g, int threshold, Func<int[],int> heuristic, int[] goal, HashSet<long> path, long budget) {
    int f = g + heuristic(state);
    if (f > threshold) return (null, f, 0);
    if (state.SequenceEqual(goal)) return (g, f, 0);
    if (budget <= 0) return (null, int.MaxValue, 0);
    long nTotal = 1; int nextT = int.MaxValue; int b = Array.IndexOf(state, 0);
    foreach (int n in Neighbors(b)) {
        int[] s2 = Move(state, b, n);
        long e2 = Encode(s2);
        if (path.Contains(e2)) continue;
        path.Add(e2);
        var (res, t, nn) = Dfs(s2, g+1, threshold, heuristic, goal, path, budget - nTotal);
        nTotal += nn; path.Remove(e2);
        if (res.HasValue) return (res, t, nTotal);
        if (t < nextT) nextT = t;
        if (nTotal >= budget) return (null, nextT, nTotal);
    }
    return (null, nextT, nTotal);
}

// Sanity préliminaire : Manhattan est admissible et résout une instance facile.
int[] easy = Scramble(GOAL, 8, seed: 1);
var (sm, _) = IdaStar(easy, Manhattan);
Console.WriteLine($"Sanity : instance facile optimal={sm} (Manhattan résout bien)");

Sanity : instance facile optimal=8 (Manhattan résout bien)


## 8. Heuristique additive et vérification d'admissibilité

`h_additive(s) = Σ PDB_groupe(s)`. On **vérifie empiriquement l'admissibilité** :
sur un échantillon d'instances résolues à l'optimum (via Manhattan, elle-même
admissible), `h_additive(s) ≤ optimal(s)` doit toujours tenir. On vérifie aussi la
convergence (même optimum) sur une instance facile.

In [8]:
int HAdditive(int[] state) => ADD_TBLS.Zip(GROUPS).Sum(p => PdbLookup(state, p.First, p.Second));

// Convergence : deux heuristiques admissibles -> même optimum.
var (sa, _) = IdaStar(easy, HAdditive);
Console.WriteLine($"Convergence confirmée : additive optimal={sa} == Manhattan optimal={sm}");

// Admissibilité empirique : h_additive <= optimal (optimal via Manhattan-IDA*).
var rngAdm = new Random(7); int checkedCount = 0; int violations = 0;
for (int i=0; i<40; i++) {
    int[] s = Scramble(GOAL, 14, seed: rngAdm.Next(0, 1_000_000_000));
    var (opt, _) = IdaStar(s, Manhattan);
    if (!opt.HasValue) continue;
    if (HAdditive(s) > opt.Value) violations++;
    checkedCount++;
}
Console.WriteLine($"Admissibilité : {checkedCount} instances, h_additive <= optimal tient {checkedCount - violations}/{checkedCount} fois");
Console.WriteLine($"Sur l'instance de démo : Manhattan={Manhattan(INST)}, h_additive={HAdditive(INST)} (additive >= Manhattan attendu)");

Convergence confirmée : additive optimal=8 == Manhattan optimal=8


Admissibilité : 40 instances, h_additive <= optimal tient 40/40 fois


Sur l'instance de démo : Manhattan=26, h_additive=28 (additive >= Manhattan attendu)


## 6b. Noyau : partage de coûts (cost partitioning) — version C#

Cette section est le pendant C# de la section 6b du notebook Python.
On y nomme le **partage de coûts** (cost partitioning, Edelkamp & Schrödl
2012, ch. 8) comme le noyau qui rend l'addition admissible des sections 4
à 6 (Korf & Felner 2002) — chaque coup du taquin déplace une tuile, qui
appartient à un seul groupe de la partition 4-4-4-3, donc le coût 1 du
coup est affecté *entièrement* à ce groupe et l'addition reste admissible.

Le **partage saturé** (Felner, Korf, Hanan & Adelman 2004 ; Edelkamp &
Schrödl 2012, § 8) en est une généralisation non triviale : on borne
chaque contribution à `bound / N_GROUPS` avant de sommer, ce qui reste
admissible tant que `bound ≥ optimal(s)` sur l'état considéré. Sur la
partition 4-4-4-3 équilibrée du 15-puzzle, l'écart numérique avec
l'additive est faible (les maxima 21, 17, 19, 16 dépassent rarement 15)
mais le geste est le point pédagogique : *un partage de coûts n'est pas
forcément canonique*.

*Ancrage page mesuré (2026-10-08) : l'index du PDF archivé (exemplaire
tronqué, index pp. 826-835 seul lisible) ancre action cost partitioning /
pattern database partitioning à la p. 312. Numéro de chapitre (ch. 8)
cohérent avec la TOC de l'édition, à confirmer sur un exemplaire complet.*


In [9]:
// Coûts maxima par table (construits par BuildAdditivePdb ci-dessus).
// Pour une partition 4-4-4-3, on attend (21, 17, 19, 16) environ ; on les relit.
int[] MAX_PER_PDB = ADD_TBLS.Select(t => (int)t.Max()).ToArray();
int N_GROUPS = ADD_TBLS.Length;
Console.WriteLine($"Max par PDB : [{string.Join(", ", MAX_PER_PDB)}]");

int HMax(int[] state) =>
    ADD_TBLS.Zip(GROUPS, (t, g) => PdbLookup(state, t, g)).Max();

// Heuristique partage saturé (Felner, Korf, Hanan & Adelman 2004) :
// borne chaque contribution à bound / N_GROUPS avant de sommer.
// Admissible par minoration terme à terme, pas parce que bound
// dépasserait l'optimum : le diamètre du 15-puzzle est 80 (God's number)
// et bound=60 est en dessous. min(v, perGroup) <= v, donc
// HSaturated <= HAdditive <= h* -- le cap resserre, il ne gonfle pas.
int HSaturated(int[] state, int bound = 60) {
    int perGroup = bound / N_GROUPS;
    return ADD_TBLS.Zip(GROUPS, (t, g) => Math.Min(PdbLookup(state, t, g), perGroup)).Sum();
}

// Comparaison sur l'instance de démo (INST définie plus haut dans le notebook).
Console.WriteLine($"Sur INST : HMax       = {HMax(INST),3}");
Console.WriteLine($"Sur INST : HAdditive  = {HAdditive(INST),3}");
Console.WriteLine($"Sur INST : HSaturated = {HSaturated(INST),3} (per_group={60 / N_GROUPS})");
Console.WriteLine($"Vérif : HMax <= HAdditive (toujours ?) {HMax(INST) <= HAdditive(INST)}");
Console.WriteLine($"Vérif : HSaturated <= HAdditive (cap) {HSaturated(INST) <= HAdditive(INST)}");


Max par PDB : [21, 17, 19, 16]


Sur INST : HMax       =  11


Sur INST : HAdditive  =  28


Sur INST : HSaturated =  28 (per_group=15)


Vérif : HMax <= HAdditive (toujours ?) True


Vérif : HSaturated <= HAdditive (cap) True


### Lecture du résultat : ce que la comparaison HMax / HAdditive / HSaturated dit

Trois heuristiques, trois listes, toutes admissibles :

- `HMax` ne retient par état qu'une seule des quatre tables. C'est
  l'heuristique de **Culberson & Schaeffer (1996)** : admissible mais
  pessimale.
- `HAdditive` (Korf & Felner 2002) somme les quatre tables — la
  contribution de chaque table au partage de coûts canonique est 1 pour
  le groupe qui contient la tuile déplacée, 0 pour les autres. Strictement
  ≥ HMax, dominant Manhattan sur les instances intermédiaires.
- `HSaturated` (Felner et al. 2004) borne chaque table à `bound / N_GROUPS`
  avant de sommer. Sur la partition 4-4-4-3 équilibrée, l'écart avec
  l'additive reste petit sur les états proches du but (les lectures
  brutes dépassent rarement `bound / N_GROUPS = 15`). L'intérêt
  pédagogique est le **geste formel** : un partage de coûts peut être
  non canonique et rester admissible tant que `Σ c_i ≥ optimal(s)`.

**Ce que le cap change — et ce qu'il ne change pas ici.** Aucune sortie de
ce carnet n'exhibe un gain du saturé : sur l'instance de démo, `HSaturated`
et `HAdditive` valent tous deux 28 (sortie de la cellule précédente), le cap
n'y mord pas. C'est le cas attendu sur la partition 4-4-4-3 *équilibrée* du
15-puzzle, dont les maxima (21, 17, 19, 16) laissent la part `perGroup = 15`
rarement contraignante. L'intérêt du saturé **ici** est donc le geste formel,
pas le chiffre : un partage de coûts n'est pas forcément canonique, et le cap
est ce qui garde l'admissibilité quand une table devient assez profonde pour
que sa contribution brute dépassât la part qui lui revient.

**Note de parité C# / Python.** Les deux carnets jumeaux ne partagent pas
le même générateur pseudo-aléatoire (`System.Random` côté C# ;
`random.Random`, Mersenne Twister, côté Python) : à graine égale
(`seed=11`), l'instance de démo `INST` **diffère** d'une version à
l'autre (mesuré : `(1, 12, 0, 3, ...)` en C#, Manhattan 26 ;
`(7, 12, 11, 2, ...)` en Python, Manhattan 32). Les valeurs absolues
`HMax`, `HAdditive` et `HSaturated` ne sont donc **pas** comparables entre
les deux carnets ; ce qui l'est, ce sont les **invariants**
(`HMax <= HAdditive`, `HSaturated <= HAdditive`) et la structure de la
comparaison.

### Dominance : additive ≥ Manhattan, lues sur l'instance de démo

La sortie porte trois verdicts distincts. (1) `additive optimal=8 == Manhattan optimal=8` : sur l'instance de test, IDA* trouve le même optimum sous les deux heuristiques — l'optimalité est préservée. (2) `40/40` : sur 40 instances brouillées, h_additive ≤ optimal à chaque fois — l'admissibilité tient en pratique. (3) `Manhattan=26, h_additive=28` : sur l'instance de démo, l'additive **domine** strictement Manhattan. Le lien avec la théorie : Manhattan est exactement la PDB additive des 15 groupes d'**une** tuile — raffiner la partition (1-1-…-1 vers 4-4-4-3) ne peut que resserrer la borne. Un heuristique qui domine *et* reste admissible explore mécaniquement moins de nœuds à optimalité égale : c'est le seul « free lunch » de la recherche heuristique.

## 9. Expérience : Manhattan vs PDB additive (la valeur du moteur, visible)

Trois instances **réalistement difficiles** (brouillage 50-80 pas, optimal ~24-53
coups), choisies pour révéler la frontière de résolubilité au budget de 2M nœuds.
On résout à l'optimum avec (a) Manhattan seule, puis (b) la PDB additive.
On mesure les **nœuds explorés** : c'est la métrique qui révèle la puissance de
l'heuristique. Sur les instances intermédiaires (optimal ~45-53), **Manhattan
dépasse le budget de 2M nœuds sans conclure**, tandis que la PDB additive
(4-4-4-3) résout — assez pour repousser la frontière du résolvable.

In [10]:
var INSTANCES = new Dictionary<string,int[]> {
    ["I1 (facile, opt 24)"] = Scramble(GOAL, 50, seed: 11),
    ["I2 (moyen, opt 45)"] = Scramble(GOAL, 81, seed: 78690737),
    ["I3 (dur, opt 53)"] = Scramble(GOAL, 71, seed: 163151276),
};

// bench: name, opt, heuristics, nodes(Manhattan/additive), solved flags, speedup, times
var bench = new List<(string name, int? opt, int mh, int ha, long nm, long na, bool mSolved, bool aSolved, double speedup, double tm, double ta)>();
foreach (var (name, s) in INSTANCES) {
    sw.Restart(); var (solM, nm) = IdaStar(s, Manhattan); sw.Stop(); double tm = sw.Elapsed.TotalSeconds;
    sw.Restart(); var (solA, na) = IdaStar(s, HAdditive); sw.Stop(); double ta = sw.Elapsed.TotalSeconds;
    int? opt = solM ?? solA;
    double speedup = (solM.HasValue && solA.HasValue && na > 0) ? (double)nm / na : double.NaN;
    bench.Add((name, opt, Manhattan(s), HAdditive(s), nm, na, solM.HasValue, solA.HasValue, speedup, tm, ta));
    string nmS = solM.HasValue ? $"{nm,12:N0}" : "  >= 2M (limite)";
    string naS = solA.HasValue ? $"{na,12:N0}" : "  >= 2M (limite)";
    Console.WriteLine($"{name,-20} optimal={opt} | Manhattan: {nmS} ({tm,6:F1}s) | additive: {naS} ({ta,6:F1}s) | speedup x{speedup,6:F1}");
}

I1 (facile, opt 24)  optimal=24 | Manhattan:        2 114 (   0,0s) | additive:          732 (   0,0s) | speedup x   2,9


I2 (moyen, opt 45)   optimal=45 | Manhattan:   >= 2M (limite) (   6,7s) | additive:      299 773 (   6,9s) | speedup x   NaN


I3 (dur, opt 53)     optimal=53 | Manhattan:   >= 2M (limite) (   4,8s) | additive:      392 168 (   7,0s) | speedup x   NaN


## Synthèse : la PDB additive repousse la frontière du résolvable

(Version texte — pas de graphique matplotlib en C# ; le tableau numérique brut
montre directement le nœud où chaque heuristique termine ou dépasse le budget.)

In [11]:
Console.WriteLine($"{"Instance",-20} {"opt",-5} {"Manhattan nodes",-18} {"Additive nodes",-18} {"speedup",-10}");
Console.WriteLine(new string('-', 75));
foreach (var r in bench) {
    string nmS = r.mSolved ? r.nm.ToString("N0") : ">= 2M";
    string naS = r.aSolved ? r.na.ToString("N0") : ">= 2M";
    Console.WriteLine($"{r.name,-20} {r.opt,-5} {nmS,-18} {naS,-18} {r.speedup,-10:F1}x");
}
Console.WriteLine();
Console.WriteLine("Lecture : sur l'instance facile (I1), les deux heuristiques terminent et la");
Console.WriteLine("PDB additive explore ~3x moins de nœuds. Sur I2 et I3 (optimal 45 et 53),");
Console.WriteLine("Manhattan dépasse le budget de 2M nœuds sans conclure, tandis que la PDB");
Console.WriteLine("additive (4-4-4-3) résout à l'optimum — elle repousse la frontière du résolvable.");

Instance             opt   Manhattan nodes    Additive nodes     speedup   


---------------------------------------------------------------------------


I1 (facile, opt 24)  24    2 114              732                2,9       x


I2 (moyen, opt 45)   45    >= 2M              299 773            NaN       x


I3 (dur, opt 53)     53    >= 2M              392 168            NaN       x


Lecture : sur l'instance facile (I1), les deux heuristiques terminent et la


PDB additive explore ~3x moins de nœuds. Sur I2 et I3 (optimal 45 et 53),


Manhattan dépasse le budget de 2M nœuds sans conclure, tandis que la PDB


additive (4-4-4-3) résout à l'optimum — elle repousse la frontière du résolvable.


### Lecture du tableau : ce que « speedup NaN » veut dire

Deux lignes portent `speedup NaN`, et c'est l'information la plus importante du tableau. Sur I1 (optimal 24), les deux heuristiques terminent : 2 114 nœuds contre 732, rapport 2,9 — l'additive fait mieux, mais Manhattan *suffit*. Sur I2 et I3 (optimaux 45 et 53), Manhattan dépasse le budget de 2 M nœuds **sans conclure** : le rapport est `NaN` parce qu'on ne peut pas diviser par un échec. L'additive, elle, conclut à l'optimum en 299 773 et 392 168 nœuds. La leçon n'est pas « 3× plus vite » mais « résout l'irrésolu » : entre les deux colonnes, ce n'est pas la même classe d'instances qui est accessible. La frontière du résoluble a bougé.

## 11. Limites et au-delà

- **Coût de construction.** Plus le groupe est grand, plus la PDB est fine mais
  plus le BFS est coûteux (`P(16,k)` explose). Korf & Felner poussent aux
  partitions **6-6-3** puis **7-8** (avec symétries) pour résoudre les instances à
  ~66 coups — au prix de tables de plusieurs gigaoctets.
- **PDB symétriques.** Le 15-puzzle admet une symétrie (miroir + renversement des
  numéros) qui permet de déduire une seconde PDB gratuitement (Felner et al. 2004).
- **Généralité.** La même technique résout le Rubik's Cube (Korf 1997), la Tour de
  Hanoï, tout puzzle de permutation. C'est l'archétype du compromis
  *mémoire contre qualité d'heuristique*.

## 12. Exemples guidés — les trois résolutions créditées

Les trois sections qui suivent sont des **exemples guidés** : ce sont les
résolutions créditées de la PR étudiante [#24987](https://github.com/jsboige/CoursIA/pull/24987)
(auteure : **EmmaLehec**), reprises telles quelles. Elles montrent la méthode
sur trois sujets déjà résolus ; les **exercices à faire** sont en section 13.

Les exemples guidés 1 à 3 sont les **exercices 1 à 3 d'origine**, désormais
résolus : le code de l'étudiante y est conservé tel quel, en-tête « Exercice N »
compris. L'énoncé de chaque exercice résolu est reproduit avant la solution,
pour que le lecteur puisse reconnaître le problème avant d'en lire la réponse.


### Exemple guidé 1 — Partition plus fine (6-6-3) (crédité #24987 — EmmaLehec)

> Résolution originale : **EmmaLehec**, PR étudiante [#24987](https://github.com/jsboige/CoursIA/pull/24987). Reprise telle quelle ; seule la mise en forme est harmonisée avec les sections voisines.

**Énoncé de l'exercice résolu** — Remplacez la partition 4-4-4-3 par **6-6-3**
(Korf & Felner 2002). Combien d'états par PDB (`P(16,6)×10`, `P(16,3)×13`) ?
Mesurez le speedup supplémentaire sur les instances du banc d'essai. Attention
au temps de construction des groupes de 6.


In [12]:
// Exercice 1 — Partition plus fine (6-6-3), Korf & Felner 2002
int[][] GROUPS_663 = { new[]{1,2,3,4,5,6}, new[]{7,8,9,10,11,12}, new[]{13,14,15} };

// Étape 0 : la partition doit être EXACTEMENT disjointe et couvrir les 15 tuiles
// (sinon un même coup est compté dans deux tables -> admissibilité cassée).
var tiles663 = GROUPS_663.SelectMany(g => g).OrderBy(t => t).ToArray();
Console.WriteLine($"Partition disjointe et complète : {tiles663.SequenceEqual(Enumerable.Range(1, 15))}");
foreach (var g in GROUPS_663)
    Console.WriteLine($"  groupe ({string.Join(",", g)}) : P(16,{g.Length}) x {16 - g.Length} = {P(16, g.Length) * (16 - g.Length):N0} états attendus");

// Indice : réutilisez BuildAdditivePdb sur chaque groupe.
//   Étape 1 : construisez la liste des tables (var tbls663 = GROUPS_663.Select(g => BuildAdditivePdb(g).table).ToArray();).
//   Étape 2 : attention, un groupe de 6 -> P(16,6)*10 ~ 57M états (~57 Mo) ; le BFS prend ~1-2 min.
sw.Restart();
var tbls663 = new byte[GROUPS_663.Length][];
for (int i = 0; i < GROUPS_663.Length; i++) {
    var swG = System.Diagnostics.Stopwatch.StartNew();
    var (tbl, nSeen663) = BuildAdditivePdb(GROUPS_663[i]);
    swG.Stop();
    tbls663[i] = tbl;
    int dm663 = tbl.Where(v => v < 255).Max();
    Console.WriteLine($"  groupe ({string.Join(",", GROUPS_663[i])}) : {nSeen663:N0} états, dist max = {dm663}, {swG.Elapsed.TotalSeconds:F1}s");
}
sw.Stop();
Console.WriteLine($"PDB additives 6-6-3 construites en {sw.Elapsed.TotalSeconds:F1}s");

// Heuristique additive 6-6-3 : somme des 3 tables (groupes disjoints).
int HAdditive663(int[] state) => tbls663.Zip(GROUPS_663).Sum(p => PdbLookup(state, p.First, p.Second));

// Étape 3 : vérifications (mêmes protocoles qu'au §8).
var (s663easy, _) = IdaStar(easy, HAdditive663);
Console.WriteLine($"Convergence : 6-6-3 optimal={s663easy} == Manhattan optimal={sm}");

var rngAdm663 = new Random(7); int checked663 = 0, viol663 = 0, dom663 = 0;
for (int i = 0; i < 40; i++) {
    int[] st = Scramble(GOAL, 14, seed: rngAdm663.Next(0, 1_000_000_000));
    var (opt, _) = IdaStar(st, Manhattan);
    if (!opt.HasValue) continue;
    if (HAdditive663(st) > opt.Value) viol663++;
    if (HAdditive663(st) >= HAdditive(st)) dom663++;
    checked663++;
}
Console.WriteLine($"Admissibilité : h_663 <= optimal tient {checked663 - viol663}/{checked663} fois");
Console.WriteLine($"h_663 >= h_4443 sur {dom663}/{checked663} instances (pas de dominance garantie : 6-6-3 n'affine pas 4-4-4-3)");
Console.WriteLine($"Instance de démo : Manhattan={Manhattan(INST)}, h_4443={HAdditive(INST)}, h_663={HAdditive663(INST)}");

// Étape 4 : banc d'essai — on réutilise les mesures Manhattan / 4-4-4-3 de `bench` (§9).
Console.WriteLine();
Console.WriteLine($"{"Instance",-20} {"opt",-4} {"h M/4443/663",-14} {"Manhattan",-11} {"4-4-4-3",-10} {"6-6-3",-10} {"vs 4443",-9} {"vs Manh.",-9}");
Console.WriteLine(new string('-', 92));
foreach (var r in bench) {
    int[] st = INSTANCES[r.name];
    sw.Restart(); var (sol663, n663) = IdaStar(st, HAdditive663); sw.Stop();
    string nmS = r.mSolved ? r.nm.ToString("N0") : ">= 2M";
    string naS = r.aSolved ? r.na.ToString("N0") : ">= 2M";
    string n6S = sol663.HasValue ? n663.ToString("N0") : ">= 2M";
    double vs4443 = (r.aSolved && sol663.HasValue) ? (double)r.na / n663 : double.NaN;
    double vsM = (r.mSolved && sol663.HasValue) ? (double)r.nm / n663 : double.NaN;
    string hS = $"{r.mh}/{r.ha}/{HAdditive663(st)}";
    Console.WriteLine($"{r.name,-20} {sol663,-4} {hS,-14} {nmS,-11} {naS,-10} {n6S,-10} {("x" + vs4443.ToString("F1")),-9} {("x" + vsM.ToString("F1")),-9}");
}


Partition disjointe et complète : True


  groupe (1,2,3,4,5,6) : P(16,6) x 10 = 57 657 600 états attendus


  groupe (7,8,9,10,11,12) : P(16,6) x 10 = 57 657 600 états attendus


  groupe (13,14,15) : P(16,3) x 13 = 43 680 états attendus


  groupe (1,2,3,4,5,6) : 57 657 600 états, dist max = 28, 327,8s


  groupe (7,8,9,10,11,12) : 57 657 600 états, dist max = 27, 354,0s


... (7 outputs left out)

--------------------------------------------------------------------------------------------


I1 (facile, opt 24)  24   16/20/20       2 114       732        262        x2,8      x8,1     


I2 (moyen, opt 45)   45   31/31/31       >= 2M       299 773    113 112    x2,7      xNaN     


I3 (dur, opt 53)     53   41/43/45       >= 2M       392 168    41 097     x9,5      xNaN     


### Conclusion — Exemple guidé 1

6-6-3 : 2 × 57 657 600 + 43 680 états, 569,6 s de construction (contre 9,2 s pour la 4-4-4-3). L'heuristique reste admissible (40/40). Speedup par rapport à 4-4-4-3 : ×2,8 (I1), ×2,7 (I2), ×9,5 (I3). On paie beaucoup plus de mémoire et de temps de construction, mais la recherche devient nettement plus courte, surtout sur l'instance la plus dure.


### Attendus — Exemple guidé 1 (partition 6-6-3)

Ce que vous devez observer si votre implémentation est juste : deux PDB de P(16,6) = 5 765 760 arrangements × 10 = 57 657 600 états chacune, une de P(16,3) × 13 = 43 680 — le temps de construction passe de 9,2 s (4-4-4-3) à 569,6 s (6-6-3) (d'où l'avertissement de l'énoncé), les distances max par groupe montent (un groupe de 6 est plus mal plaçable qu'un groupe de 4), et h_additive resserre encore : attendez-vous à ce que I1 descende nettement sous les 732 nœuds de la 4-4-4-3. Anti-piège : la partition doit rester **exactement** disjointe (15 tuiles = 6+6+3) ; réutiliser une tuile dans deux groupes compte ses coups deux fois et casse l'admissibilité — la cellule de vérification du §8 doit rester 40/40.

### Exemple guidé 2 — PDB symétrique : attention, toutes les symétries ne se valent pas (Felner et al. 2004) (crédité #24987 — EmmaLehec)

> Résolution originale : **EmmaLehec**, PR étudiante [#24987](https://github.com/jsboige/CoursIA/pull/24987). Reprise telle quelle ; seule la mise en forme est harmonisée avec les sections voisines.

**Énoncé de l'exercice résolu** — Une transformation $\sigma$ des états donne une seconde heuristique **gratuite** si elle vérifie **deux** conditions :
1. **préserver les distances** : $\text{dist}(s, g) = \text{dist}(\sigma(s), \sigma(g))$ pour tous $s, g$ ;
2. **laisser le but invariant** : $\sigma(g_{\text{but}}) = g_{\text{but}}$ — sinon $h(\sigma(s)) = \text{dist}(\sigma(s), g_{\text{but}})$ n'est pas la distance vers le vrai but, et l'heuristique peut **surestimer**.

La transformation « miroir vertical + renommage $t \to 16 - t$ » de l'énoncé initial est réversible mais viole la condition 2 : le but transformé n'est pas le but. Le contre-exemple mesuré plus bas (sur la grille **déjà résolue**, $h$ doit valoir $0$ et renvoie $51$) montre qu'elle n'est **pas admissible**. Un test sur quelques instances ne prouve pas l'admissibilité en général — c'est ici la condition 2, vérifiable algébriquement, qui la réfute.

La **réflexion diagonale** de Felner et al. (la transposée de la grille) vérifie les deux conditions : le but symétrique reste le but, et le renommage se fait **par position dans le but** (chaque tuile est remplacée par la tuile qui occupe la position miroir de sa position-cible), ce qui préserve exactement les distances de Manhattan et celles de la PDB.

La solution implémente la réflexion diagonale `Reflect(state)` (avec ce renommage par position-cible) : elle vérifie que la transformation laisse le but inchangé et que `HAdditive(state) == HAdditive(Reflect(state))`, puis combine les deux heuristiques par `Math.Max`. Le contre-exemple de la méthode 1 (miroir + $t \to 16-t$) est conservé dans la solution : il illustre pourquoi la condition 2 n'est pas optionnelle.

In [13]:
// ---------- Méthode 1 : la transformation de l'énoncé (miroir vertical + t -> 16 - t) ----------
Console.WriteLine("=== Méthode 1 (énoncé : miroir vertical + t -> 16 - t) ===");

int[] Reflect1(int[] state)
{
    int[] result = new int[state.Length];

    // Miroir vertical
    for (int r = 0; r < N; r++)
    {
        for (int c = 0; c < N; c++)
        {
            int source = r * N + c;
            int destination = r * N + (N - 1 - c);
            result[destination] = state[source];
        }
    }

    // Renversement des numéros
    for (int i = 0; i < result.Length; i++)
    {
        if (result[i] != 0)
        {
            result[i] = 16 - result[i];
        }
    }
    return result;
}

// Vérification : Reflect1 est son propre inverse
int[] reflected1 = Reflect1(INST);
int[] reflectedTwice1 = Reflect1(reflected1);

Console.WriteLine(
    $"Reflect1(Reflect1(s)) == s : {reflectedTwice1.SequenceEqual(INST)}"
);

// Vérification de l'heuristique
int hOriginal = HAdditive(INST);
int hReflected1 = HAdditive(Reflect1(INST));

Console.WriteLine($"HAdditive(s) = {hOriginal}");
Console.WriteLine($"HAdditive(Reflect1(s)) = {hReflected1}");
Console.WriteLine($"Égalité = {hOriginal == hReflected1}");

// PDB symétrique
int HSymmetric1(int[] state)
{
    return Math.Max(
        HAdditive(state),
        HAdditive(Reflect1(state))
    );
}

// Test final
Console.WriteLine($"HSymmetric1(INST) = {HSymmetric1(INST)}");

// Vérifications de la symétrie
Console.WriteLine($"Reflect1(GOAL) == GOAL : {Reflect1(GOAL).SequenceEqual(GOAL)}");      // False : le but n'est pas conservé
Console.WriteLine($"HSymmetric1(GOAL) = {HSymmetric1(GOAL)} (attendu 0)");              // > 0 : surestimation
Console.WriteLine($"Manhattan(s) == Manhattan(Reflect1(s)) : {Manhattan(INST) == Manhattan(Reflect1(INST))}"); // False

// ---------- Méthode 2 : réflexion par rapport à la diagonale (Felner et al.) ----------
Console.WriteLine();
Console.WriteLine("=== Méthode 2 (réflexion diagonale, Felner et al.) ===");

int[] Reflect2(int[] state)
{
    int[] result = new int[state.Length];
    for (int r = 0; r < N; r++)
    {
        for (int c = 0; c < N; c++)
        {
            int t = state[r * N + c];
            // La tuile t a pour place (gr, gc) dans le but ; elle est renommée en la tuile
            // dont la place est (gc, gr). La case vide reste la case vide.
            int renamed = (t == 0) ? 0 : ((t - 1) % N) * N + (t - 1) / N + 1;
            result[c * N + r] = renamed;   // miroir par rapport à la diagonale
        }
    }
    return result;
}

// Vérification : Reflect2 est son propre inverse
int[] reflected2 = Reflect2(INST);
int[] reflectedTwice2 = Reflect2(reflected2);

Console.WriteLine(
    $"Reflect2(Reflect2(s)) == s : {reflectedTwice2.SequenceEqual(INST)}"
);

// Vérification de l'heuristique
int hReflected2 = HAdditive(Reflect2(INST));

Console.WriteLine($"HAdditive(s) = {hOriginal}");
Console.WriteLine($"HAdditive(Reflect2(s)) = {hReflected2}");
Console.WriteLine($"Égalité = {hOriginal == hReflected2}");

// PDB symétrique
int HSymmetric2(int[] state)
{
    return Math.Max(
        HAdditive(state),
        HAdditive(Reflect2(state))
    );
}

// Test final
Console.WriteLine($"HSymmetric2(INST) = {HSymmetric2(INST)}");

// Vérifications de la symétrie
Console.WriteLine($"Reflect2(GOAL) == GOAL : {Reflect2(GOAL).SequenceEqual(GOAL)}");      // doit afficher True
Console.WriteLine($"HSymmetric2(GOAL) = {HSymmetric2(GOAL)} (attendu 0)");
Console.WriteLine($"Manhattan(s) == Manhattan(Reflect2(s)) : {Manhattan(INST) == Manhattan(Reflect2(INST))}"); // True

=== Méthode 1 (énoncé : miroir vertical + t -> 16 - t) ===


Reflect1(Reflect1(s)) == s : True


HAdditive(s) = 28


HAdditive(Reflect1(s)) = 35


Égalité = False


HSymmetric1(INST) = 35


... (8 outputs left out)

HSymmetric2(INST) = 28


Reflect2(GOAL) == GOAL : True


HSymmetric2(GOAL) = 0 (attendu 0)


Manhattan(s) == Manhattan(Reflect2(s)) : True


### Conclusion — Exemple guidé 2

- **Méthode 1 (énoncé : miroir vertical + t → 16 − t)** : elle est réversible, mais elle ne laisse pas le but inchangé (`Reflect1(GOAL) ≠ GOAL`). `HSymmetric1(GOAL) = 51` sur une grille déjà résolue : elle surestime, donc elle n'est **pas admissible**.
- **Méthode 2 (réflexion diagonale, Felner et al.)** : le but reste inchangé (`HSymmetric2(GOAL) = 0`) et Manhattan est conservé. On obtient une seconde heuristique admissible **sans nouveau BFS**, combinée par `Math.Max`.
- Être réversible ne suffit pas : une symétrie utile doit aussi laisser le but inchangé.


### Attendus — Exemple guidé 2 (PDB symétrique)

La réflexion diagonale (transposée de la grille, renommage par position-cible) vérifie les deux conditions d'une symétrie utile : distances préservées **et** but invariant. Vous obtenez une **seconde heuristique sans nouveau BFS**, combinée par `Math.Max` — admissible car chacune l'est, strictement meilleure dès qu'une des deux voit mieux.

Si votre `Reflect` laisse le but transformé différent du but (par exemple avec un renommage arbitraire des numéros comme $t \to 16 - t$), l'heuristique combinée surestime sur certaines instances — sur la grille résolue $h$ doit valoir $0$ ; toute autre valeur trahit une transformation qui ne fixe pas le but. Le contre-exemple mesuré dans la solution (méthode 1) illustre ce piège.

### Exemple guidé 3 — Généralisation au 24-puzzle (crédité #24987 — EmmaLehec)

> Résolution originale : **EmmaLehec**, PR étudiante [#24987](https://github.com/jsboige/CoursIA/pull/24987). Reprise telle quelle ; seule la mise en forme est harmonisée avec les sections voisines.

**Énoncé de l'exercice résolu** — Le **24-puzzle** (5×5) est le vrai défi : Manhattan y est désespérément faible et
seules les PDB additives rendent la résolution optimale envisageable. Adaptez ce
notebook (partition 6-6-6-7) et résolvez une instance. Que
constatez-vous sur le rapport construction/qualité ?

In [14]:
// Exercice 3 — Généralisation au 24-puzzle (5x5)
// Indice : la structure est inchangée ; changent la taille (N=5), le but (1..24,0),
// et la partition.
//   Étape 1 : redéfinissez GOAL24 et Neighbors pour N=5.
//   Étape 2 : attention à l'explosion de P(25, k) — restez sur des groupes <= 6.

// ---------- Étape 1 : le 24-puzzle (5x5) ----------
const int N5 = 5;              // côté de la grille
const int CELLS5 = N5 * N5;    // 25 cases

int[] GOAL24 = Enumerable.Range(1, 24).Append(0).ToArray();   // tuiles 1..24, case vide en bas à droite

int[] Neighbors24(int cell) {
    int r = cell / N5, c = cell % N5;
    var res = new List<int>();
    if (r > 0) res.Add(cell - N5);
    if (r < N5 - 1) res.Add(cell + N5);
    if (c > 0) res.Add(cell - 1);
    if (c < N5 - 1) res.Add(cell + 1);
    return res.ToArray();
}
// Voisins précalculés une fois pour toutes (la recherche les consulte des millions de fois).
int[][] NEIGH24 = Enumerable.Range(0, CELLS5).Select(c => Neighbors24(c)).ToArray();

int Manhattan24(int[] state) {
    int h = 0;
    for (int cell = 0; cell < CELLS5; cell++) {
        int t = state[cell];
        if (t == 0) continue;
        int goalCell = t - 1;   // dans GOAL24, la tuile t est rangée en case t-1
        h += Math.Abs(goalCell / N5 - cell / N5) + Math.Abs(goalCell % N5 - cell % N5);
    }
    return h;
}

// Même marche aléatoire que Scramble (section 3.b), sur la grille 5x5 (Move est réutilisé tel quel).
int[] Scramble24(int nSteps, int seed) {
    var rng = new Random(seed);
    int[] s = (int[])GOAL24.Clone();
    int[] prev = null;
    for (int i = 0; i < nSteps; i++) {
        int b = Array.IndexOf(s, 0);
        var cands = NEIGH24[b].Where(n => !Move(s, b, n).SequenceEqual(prev ?? Array.Empty<int>())).ToList();
        int nxt = cands[rng.Next(cands.Count)];
        prev = (int[])s.Clone();
        s = Move(s, b, nxt);
    }
    return s;
}

Console.WriteLine($"But 24 : Manhattan = {Manhattan24(GOAL24)} ; voisins coin = {NEIGH24[0].Length}, bord = {NEIGH24[2].Length}, centre = {NEIGH24[12].Length}");
var demo24 = Scramble24(30, 1);
Console.WriteLine($"Instance 5x5 (30 pas, seed 1) : ({string.Join(", ", demo24)}) ; Manhattan = {Manhattan24(demo24)}");

But 24 : Manhattan = 0 ; voisins coin = 2, bord = 3, centre = 4


Instance 5x5 (30 pas, seed 1) : (6, 2, 4, 5, 9, 7, 1, 3, 14, 10, 11, 12, 0, 8, 18, 16, 17, 19, 20, 15, 21, 22, 23, 13, 24) ; Manhattan = 22


In [15]:
// ---------- Étape 2 : l'explosion de P(25, k) ----------
// Même formule qu'au 15-puzzle : P(25,k) placements des k tuiles × (25-k) positions de la case vide.
// On calcule en long : dès k = 6 le résultat dépasse la capacité d'un int.
long PLong(int n, int k) { long r = 1; for (int i = 0; i < k; i++) r *= (n - i); return r; }
long PdbSize24(int k) => PLong(CELLS5, k) * (CELLS5 - k);

Console.WriteLine("Taille d'une PDB du 24-puzzle (1 octet par état abstrait) :");
foreach (int k in new[] { 3, 4, 5, 6, 7 })
    Console.WriteLine($" groupe de {k} tuiles : P(25,{k}) x {CELLS5 - k,2} = {PdbSize24(k),18:N0} états ≈ {PdbSize24(k) / 1e9,6:F2} Go");

long total6667 = 3 * PdbSize24(6) + PdbSize24(7);
Console.WriteLine($"Partition 6-6-6-7 demandée : {total6667:N0} états ≈ {total6667 / 1e9:F1} Go de RAM");
Console.WriteLine($"Plus grand tableau autorisé par .NET : Array.MaxLength = {Array.MaxLength:N0} cases");
Console.WriteLine($"=> un seul groupe de 6 ({PdbSize24(6):N0} états) dépasse déjà cette limite : 6-6-6-7 est infaisable tel quel sur un PC.");
Console.WriteLine("=> on garde la structure du notebook mais avec des groupes plus petits (3, 4, voire 5 tuiles), et on mesure le compromis.");

Taille d'une PDB du 24-puzzle (1 octet par état abstrait) :


 groupe de 3 tuiles : P(25,3) x 22 =            303 600 états ≈   0,00 Go


 groupe de 4 tuiles : P(25,4) x 21 =          6 375 600 états ≈   0,01 Go


 groupe de 5 tuiles : P(25,5) x 20 =        127 512 000 états ≈   0,13 Go


 groupe de 6 tuiles : P(25,6) x 19 =      2 422 728 000 états ≈   2,42 Go


 groupe de 7 tuiles : P(25,7) x 18 =     43 609 104 000 états ≈  43,61 Go


Partition 6-6-6-7 demandée : 50 877 288 000 états ≈ 50,9 Go de RAM


Plus grand tableau autorisé par .NET : Array.MaxLength = 2 147 483 591 cases


=> un seul groupe de 6 (2 422 728 000 états) dépasse déjà cette limite : 6-6-6-7 est infaisable tel quel sur un PC.


=> on garde la structure du notebook mais avec des groupes plus petits (3, 4, voire 5 tuiles), et on mesure le compromis.


In [16]:
// ---------- Étape 3 : construire les PDB additives du 24-puzzle ----------

// Rang compact SANS allocation (PermRank/RankAbstract créent des listes à chaque appel : trop lent
// pour des millions d'états). Même principe : chiffre i = position de la i-ème tuile parmi les cases
// encore libres (base 25-i), dernier chiffre = case vide parmi les 25-k cases restantes.
long Rank24(int[] pos, int k, int blank) {
    long r = 0;
    for (int i = 0; i < k; i++) {
        int smaller = 0;
        for (int j = 0; j < i; j++) if (pos[j] < pos[i]) smaller++;
        r = r * (CELLS5 - i) + (pos[i] - smaller);
    }
    int blankSmaller = 0;
    for (int j = 0; j < k; j++) if (pos[j] < blank) blankSmaller++;
    return r * (CELLS5 - k) + (blank - blankSmaller);
}

// Test exhaustif pour k = 3 : chaque état abstrait doit recevoir un rang distinct dans [0, taille).
{
    var seenRank = new bool[PdbSize24(3)]; int collisions = 0, horsBornes = 0, nb = 0;
    for (int a = 0; a < CELLS5; a++) for (int b = 0; b < CELLS5; b++) for (int c = 0; c < CELLS5; c++) for (int v = 0; v < CELLS5; v++) {
        if (a == b || a == c || b == c || v == a || v == b || v == c) continue;
        long rk = Rank24(new[] { a, b, c }, 3, v); nb++;
        if (rk < 0 || rk >= seenRank.Length) { horsBornes++; continue; }
        if (seenRank[rk]) collisions++; else seenRank[rk] = true;
    }
    Console.WriteLine($"Rang24 : {nb:N0} états abstraits (k=3), {collisions} collision, {horsBornes} hors bornes (attendu 0 et 0)");
}

// Un état abstrait (positions des k tuiles + case vide) tient dans un long : 5 bits par case (0..24).
long Pack24(int[] pos, int k, int blank) { long c = blank; for (int i = 0; i < k; i++) c |= (long)pos[i] << (5 * (i + 1)); return c; }
int Unpack24(long code, int[] pos, int k) { for (int i = 0; i < k; i++) pos[i] = (int)((code >> (5 * (i + 1))) & 31); return (int)(code & 31); }

// 0-1 BFS rétrograde, comme BuildAdditivePdb : déplacer une tuile du groupe coûte 1, déplacer la
// case vide à travers un joker coûte 0. Traitement couche par couche : "cur" = distance d, "next" = d+1.
// Si un état est retrouvé plus tard avec un coût plus faible, on le corrige (vrai plus court chemin).
byte[] BuildPdb24(int[] group) {
    int k = group.Length;
    long size = PdbSize24(k);
    if (size > Array.MaxLength)
        throw new InvalidOperationException($"Groupe de {k} tuiles : {size:N0} états, trop gros pour un tableau .NET.");
    var dist = new byte[size];
    Array.Fill(dist, (byte)255);                      // 255 = pas encore atteint
    int[] pos = group.Select(t => Array.IndexOf(GOAL24, t)).ToArray();
    int blank = Array.IndexOf(GOAL24, 0);
    dist[Rank24(pos, k, blank)] = 0;
    var cur = new Queue<long>();
    var next = new Queue<long>();
    cur.Enqueue(Pack24(pos, k, blank));
    int d = 0;
    while (cur.Count > 0) {
        while (cur.Count > 0) {
            int b = Unpack24(cur.Dequeue(), pos, k);
            if (dist[Rank24(pos, k, b)] != d) continue;   // entrée périmée : déjà atteinte moins cher
            foreach (int n in NEIGH24[b]) {
                int ti = Array.IndexOf(pos, n);            // la case n contient-elle une tuile du groupe ?
                int w = ti >= 0 ? 1 : 0;
                if (ti >= 0) pos[ti] = b;                  // la tuile glisse dans l'ancienne case vide
                long rk = Rank24(pos, k, n);
                if (d + w < dist[rk]) {
                    dist[rk] = (byte)(d + w);
                    (w == 0 ? cur : next).Enqueue(Pack24(pos, k, n));
                }
                if (ti >= 0) pos[ti] = n;                  // on annule avant de tester le voisin suivant
            }
        }
        (cur, next) = (next, cur);
        d++;
    }
    return dist;
}

// Heuristique additive : somme des tables (groupes disjoints => admissible, cf. section 6).
int[] POS_OF24 = new int[CELLS5];
int[] PAT_BUF24 = new int[8];
int HAdditive24(int[] state, int[][] groups, byte[][] tables) {
    for (int c = 0; c < CELLS5; c++) POS_OF24[state[c]] = c;
    int h = 0;
    for (int g = 0; g < groups.Length; g++) {
        int k = groups[g].Length;
        for (int i = 0; i < k; i++) PAT_BUF24[i] = POS_OF24[groups[g][i]];
        h += tables[g][Rank24(PAT_BUF24, k, POS_OF24[0])];
    }
    return h;
}

// Partitions (groupes compacts, en blocs sur la grille du but) :
//    1  2  3  4  5
//    6  7  8  9 10
//   11 12 13 14 15
//   16 17 18 19 20
//   21 22 23 24  _
int[][] GROUPS24_3 = { new[]{1,2,6}, new[]{3,4,5}, new[]{7,8,9}, new[]{10,15,20}, new[]{11,12,16}, new[]{13,14,19}, new[]{17,18,22}, new[]{21,23,24} };
int[][] GROUPS24_4 = { new[]{1,2,6,7}, new[]{3,4,8,9}, new[]{5,10,15,20}, new[]{11,12,16,17}, new[]{13,14,18,19}, new[]{21,22,23,24} };
int[][] GROUPS24_5 = { new[]{1,2,3,6,7}, new[]{4,5,8,9,10}, new[]{11,12,16,17,21}, new[]{13,14,15,18,19}, new[]{20,22,23,24} };
// Bonus : la partition 5-5-5-5-4 demande ~0,5 Go de RAM et plusieurs minutes de construction.
bool AVEC_5 = true;

bool IsPartition24(int[][] groups) => groups.SelectMany(g => g).OrderBy(t => t).SequenceEqual(Enumerable.Range(1, 24));

var partitions24 = new List<(string name, int[][] groups)> { ("3x8", GROUPS24_3), ("4x6", GROUPS24_4) };
if (AVEC_5) partitions24.Add(("5-5-5-5-4", GROUPS24_5));

var PDB24 = new List<(string name, int[][] groups, byte[][] tables, long states, double sec)>();
foreach (var (name, groups) in partitions24) {
    if (!IsPartition24(groups)) throw new InvalidOperationException($"{name} n'est pas une partition disjointe de 1..24");
    var swB = System.Diagnostics.Stopwatch.StartNew();
    byte[][] tables = groups.Select(g => BuildPdb24(g)).ToArray();
    swB.Stop();
    long states = groups.Sum(g => PdbSize24(g.Length));
    bool complet = tables.All(t => Array.IndexOf(t, (byte)255) < 0);
    PDB24.Add((name, groups, tables, states, swB.Elapsed.TotalSeconds));
    Console.WriteLine($"PDB {name,-10}: {states,13:N0} états (~{states / 1e6:F0} Mo), construite en {swB.Elapsed.TotalSeconds,6:F1} s, " +
                      $"dist max par groupe = [{string.Join(", ", tables.Select(t => t.Max()))}], tous les états atteints : {complet}, h(but) = {HAdditive24(GOAL24, groups, tables)}");
}

Rang24 : 303 600 états abstraits (k=3), 0 collision, 0 hors bornes (attendu 0 et 0)


PDB 3x8       :     2 428 800 états (~2 Mo), construite en    1,0 s, dist max par groupe = [20, 21, 17, 20, 18, 15, 18, 21], tous les états atteints : True, h(but) = 0


PDB 4x6       :    38 253 600 états (~38 Mo), construite en   19,2 s, dist max par groupe = [25, 23, 27, 23, 22, 27], tous les états atteints : True, h(but) = 0


PDB 5-5-5-5-4 :   516 423 600 états (~516 Mo), construite en  451,8 s, dist max par groupe = [31, 30, 30, 28, 28], tous les états atteints : True, h(but) = 0


In [17]:
// ---------- Étape 4 : IDA* sur le 24-puzzle et résolution d'instances ----------
// Même IDA* qu'en section 7, adapté au 5x5 : l'état est modifié sur place (échange puis annulation)
// au lieu d'être recopié, et on interdit seulement de défaire le coup précédent (un long ne peut
// plus encoder 25 cases × 5 bits pour l'ensemble "path").
(int? cost, long nodes) IdaStar24(int[] start, Func<int[], int> heuristic, long nodeLimit = 5_000_000) {
    int[] s = (int[])start.Clone();
    long nodes = 0;
    int found = -1;
    const int FOUND = -1;

    int Dfs(int g, int threshold, int blank, int prevBlank) {
        int h = heuristic(s);
        int f = g + h;
        if (f > threshold) return f;
        if (h == 0 && s.SequenceEqual(GOAL24)) { found = g; return FOUND; }
        if (nodes >= nodeLimit) return int.MaxValue;
        nodes++;
        int nextT = int.MaxValue;
        foreach (int n in NEIGH24[blank]) {
            if (n == prevBlank) continue;             // ne pas défaire le coup qu'on vient de jouer
            (s[blank], s[n]) = (s[n], s[blank]);
            int t = Dfs(g + 1, threshold, n, blank);
            (s[blank], s[n]) = (s[n], s[blank]);
            if (t == FOUND) return FOUND;
            if (t < nextT) nextT = t;
            if (nodes >= nodeLimit) return nextT;
        }
        return nextT;
    }

    int th = heuristic(s);
    int blank0 = Array.IndexOf(s, 0);
    while (true) {
        int t = Dfs(0, th, blank0, -1);
        if (t == FOUND) return (found, nodes);
        if (t == int.MaxValue || nodes >= nodeLimit) return (null, nodes);
        th = t;
    }
}

// Admissibilité empirique (comme en section 8) : h_additive <= optimal sur des instances faciles.
{
    var rngAdm24 = new Random(7); int nbOk = 0, viol = 0;
    for (int i = 0; i < 30; i++) {
        int[] s = Scramble24(16, rngAdm24.Next(0, 1_000_000_000));
        var (opt, _) = IdaStar24(s, Manhattan24);
        if (!opt.HasValue) continue;
        foreach (var p in PDB24) if (HAdditive24(s, p.groups, p.tables) > opt.Value) viol++;
        nbOk++;
    }
    Console.WriteLine($"Admissibilité : {nbOk} instances x {PDB24.Count} partitions, {viol} violation (attendu 0)");
}

// Banc d'essai : de plus en plus mélangé. Budget : 5 millions de nœuds par résolution.
var INSTANCES24 = new (string name, int[] s)[] {
    ("J1 (30 pas, seed 1)",  Scramble24(30, 1)),
    ("J2 (60 pas, seed 2)",  Scramble24(60, 2)),
    ("J3 (80 pas, seed 3)",  Scramble24(80, 3)),
    ("J4 (70 pas, seed 7)",  Scramble24(70, 7)),
    ("J5 (100 pas, seed 5)", Scramble24(100, 5)),
};

var heuristics24 = new List<(string name, Func<int[], int> h)> { ("Manhattan", (Func<int[], int>)Manhattan24) };
foreach (var p in PDB24) { var pp = p; heuristics24.Add(($"PDB {pp.name}", (Func<int[], int>)(st => HAdditive24(st, pp.groups, pp.tables)))); }

var bench24 = new List<(string inst, string heur, int h0, int? opt, long nodes, double sec)>();
foreach (var (iname, s) in INSTANCES24) {
    Console.WriteLine($"{iname} :");
    foreach (var (hname, h) in heuristics24) {
        var swI = System.Diagnostics.Stopwatch.StartNew();
        var (opt, nodes) = IdaStar24(s, h);
        swI.Stop();
        bench24.Add((iname, hname, h(s), opt, nodes, swI.Elapsed.TotalSeconds));
        string res = opt.HasValue ? $"optimal = {opt,3}, {nodes,12:N0} nœuds" : $"NON RÉSOLU, budget de {nodes:N0} nœuds épuisé";
        Console.WriteLine($"   {hname,-15} h(départ) = {h(s),3} | {res} ({swI.Elapsed.TotalSeconds:F1} s)");
    }
}

Admissibilité : 30 instances x 3 partitions, 0 violation (attendu 0)


J1 (30 pas, seed 1) :


   Manhattan       h(départ) =  22 | optimal =  30,        8 414 nœuds (0,0 s)


   PDB 3x8         h(départ) =  24 | optimal =  30,        2 353 nœuds (0,0 s)


   PDB 4x6         h(départ) =  26 | optimal =  30,          529 nœuds (0,0 s)


   PDB 5-5-5-5-4   h(départ) =  26 | optimal =  30,          381 nœuds (0,0 s)


... (16 outputs left out)

   Manhattan       h(départ) =  42 | NON RÉSOLU, budget de 5 000 000 nœuds épuisé (5,8 s)


   PDB 3x8         h(départ) =  44 | NON RÉSOLU, budget de 5 000 000 nœuds épuisé (7,8 s)


   PDB 4x6         h(départ) =  48 | NON RÉSOLU, budget de 5 000 000 nœuds épuisé (8,3 s)


   PDB 5-5-5-5-4   h(départ) =  48 | NON RÉSOLU, budget de 5 000 000 nœuds épuisé (9,7 s)


### Réponse — Exemple guidé 3

- **6-6-6-7 est infaisable** : environ 51 Go de RAM, et un seul groupe de 6 dépasse déjà la taille max d'un tableau .NET. On a donc comparé des partitions plus petites.
- **Les PDB rendent résoluble ce qui ne l'était pas** : Manhattan échoue sur J3 et J4 ; la PDB 4x6 les résout à l'optimum (J3 : 250 655 nœuds, J4 : 3 413 193).
- **Rapport construction/qualité** : chaque palier multiplie la mémoire totale par 13 à 16 (3x8 : 2,4 Mo → 4x6 : 38 Mo → 5-5-5-5-4 : 0,5 Go). Le gain en nœuds est fort de 3x8 à 4x6 (×15 sur J3), puis irrégulier et plus faible que le coût de 4x6 à 5-5-5-5-4 : ×7,5 sur J2, mais seulement ×1,5 sur J3 et ×1,2 sur J4.
- **Limite** : J5 (100 pas) n'est résolue par aucune version. Une vraie instance aléatoire du 24-puzzle demande des tables bien plus grosses et beaucoup plus de calcul.


### Attendus — Exemple guidé 3 (24-puzzle)

Au 24-puzzle (5 × 5), Manhattan est désespérément loose : les instances aléatoires moyennes exigent ~100 coups, et IDA* sous Manhattan explose au-delà de tout budget raisonnable. La structure du code est inchangée (N=5, but 1..24,0) mais les tailles explosent : un groupe de 6 tuiles dans 25 cases donne P(25,6) × 19 = 2 422 728 000, soit ~2,4 milliards d'états — la table ne tient plus en mémoire sans encodage compact. C'est précisément pourquoi Korf & Felner ont poussé les partitions avec symétries et la compression : au 24-puzzle, la PDB n'est plus une optimisation, c'est la condition d'entrée.

**L'arc du notebook** : représenter (permutations), abstraire (sous-espaces), précalculer (BFS rétrograde), sommer (disjonction), vérifier (admissibilité et dominance), mesurer (frontière du résoluble). Ce pipeline est transposable à tout domaine où l'on peut payer du calcul préliminaire pour accélérer des requêtes répétées.

## 13. Exercices

> Rappel (convention de la série) : un exercice est un **squelette à compléter**.
> Les stubs s'exécutent sans erreur (`// TODO etudiant` + garde le code compilable) ;
> à vous de les remplir. Ne supprimez pas les `// Indice` / `// Étape N`.

Les trois exercices qui suivent sont **neufs** : ils ne sont pas issus des
résolutions créditées de la section 12, et aucune solution n'en est publiée. Les
exemples guidés 1 à 3 montrent la méthode sur trois sujets déjà résolus ; ces
trois exercices en sont la contrepartie non résolue.


### Exercice 4 — Partition 5-5-5 (trois groupes équilibrés)

Construisez la partition **5-5-5** et comparez-la à la 6-6-3 : coût mémoire de chaque table ($P(16,5) \times 11$ états attendus par groupe), temps de construction, admissibilité (40/40), et qualité de la guidance (nombre de nœuds explorés sur les 40 instances de la section 7). La 6-6-3 déséquilibre les groupes (un groupe de 6 est bien plus coûteux) — la 5-5-5 équilibre le compromis mémoire/guidance autrement. Où se situe l'optimum pratique ?

**Indice.** Reprenez la structure de l'**exemple guidé 1** (section 12) : un groupe de
tuiles contiguës, `BuildAdditivePdb` par groupe, puis la mesure d'admissibilité et la
mesure de guidance de la section 7. Ici, seul le découpage change : trois groupes de
cinq au lieu de 6-6-3.


In [18]:
// Exercice 4 — à compléter : partition 5-5-5 et confrontation 6-6-3
// Indice : la structure de l'exercice 1 est inchangée — GROUPS_555 =
// { new[]{1,2,3,4,5}, new[]{6,7,8,9,10}, new[]{11,12,13,14,15} }, puis
// réutilisez BuildAdditivePdb sur chaque groupe et le même protocole de
// mesure (admissibilité 40/40 + nœuds explorés) que la section 7.
// Attention : trois tables de P(16,5) x 11 états chacune — vérifiez la
// mémoire avant de lancer la construction.
// TODO etudiant
Console.WriteLine("Exercice 4 à compléter : partition 5-5-5.");

Exercice 4 à compléter : partition 5-5-5.


### Exercice 5 — Max vs somme : combiner des PDB non additives

Le PDB de l'**exemple guidé 1** (groupes 6-6-3) est **additif** (les groupes sont disjoints : la somme reste admissible). Construisez maintenant deux PDB **non additives** sur des groupes qui se chevauchent (par exemple **{1,2,3,4} et {4,5,6,7}** : la tuile 4 appartient aux deux), et combinez-les par `Math.Max`. Expliquez pourquoi la **somme** de ces deux tables casserait l'admissibilité (un coup de la tuile 4 serait compté deux fois) alors que le max reste sûr, et mesurez quelle combinaison guide le mieux.

**Pourquoi des groupes de 4 tuiles, et pas de 8.** La table d'un groupe de `k` tuiles occupe `P(16,k) * (16-k)` octets, et `BuildAdditivePdb` calcule cette taille en `int`. Pour `k = 8`, on obtient `P(16,8) * 8 = 4 151 347 200` états, **au-delà** de `int.MaxValue` (2 147 483 647) : en arithmétique `int` non contrôlée, la taille devient négative et `new byte[size]` échoue. Un groupe de 4 tuiles tient : `P(16,4) * 12 = 524 160` octets, exactement la taille de la PDB simple de la section 5. C'est le même plafond que la partition 6-6-6-7 de l'exemple guidé 1, mais ici **par addition** : au-delà de 7 tuiles par groupe, l'organe ne peut plus être appelé tel quel.


In [19]:
// Exercice 5 — à compléter : PDB non additives combinées par Max
// Indice : réutilisez BuildAdditivePdb sur des groupes qui se chevauchent
// — la table reste une vraie PDB (elle vérifie h(s) <= dist(s, but) par
// groupe), seule la COMBINAISON change : max sûr, somme fausse.
// Mesurez : h_max non-additive vs h_somme additive 4-4-4-3 sur les 40
// instances. Lequel explore le moins de nœuds à mémoire égale ?
// TODO etudiant
Console.WriteLine("Exercice 5 à compléter : combinaison max de PDB non additives.");

Exercice 5 à compléter : combinaison max de PDB non additives.


### Exercice 6 — Base de motifs sur un autre problème : les tours de Hanoï

La technique PDB ne se limite pas au taquin. Construisez une petite base de motifs pour les **tours de Hanoï** à 3 piquets et 5 disques : l'état abstrait retient la position de 2 disques choisis (les 3 autres sont « n'importe où »), la table stocke le nombre de coups restants vers l'état but. Vérifiez l'admissibilité de la table sur 20 états aléatoires (comparés au coût réel d'un BFS complet), puis mesurez la guidance : combien de nœuds A* explore-t-il avec cette table contre une heuristique triviale ?


In [20]:
// Exercice 6 — à compléter : PDB pour les tours de Hanoi (3 piquets, 5 disques)
// Indice : espace complet = 3^5 = 243 états — trivial à BFSer en entier.
// L'abstraction retient la position des disques 4 et 5 (les plus grands),
// les disques 1-3 sont projetes (n'importe quel piquet) : 3^2 = 9 entrees.
// Vérifiez h_abstraite <= dist_reelle sur 20 états (admissibilité), puis
// A* avec cette table vs A* avec h=0 : comptez les nœuds développés.
// TODO etudiant
Console.WriteLine("Exercice 6 à compléter : PDB tours de Hanoi.");

Exercice 6 à compléter : PDB tours de Hanoi.


## Conclusion

La **base de données de motifs** illustre un principe central de la recherche
heuristique avancée : **investir du calcul en amont** (précalculer des tables une
fois pour toutes) **pour rendre chaque recherche ultérieure radicalement plus
cheap**. Sur le 15-puzzle, la PDB *additive* (4-4-4-3) explore ~3 à 10x moins de
nœuds que Manhattan sur les instances résolubles, et — c'est le point décisif —
**résout à l'optimum des instances intermédiaires (optimal ~45-53 coups) où
Manhattan dépasse le budget**. Les partitions plus fines (6-6-3 puis 7-8 avec
symétries, cf exemples guidés et Korf & Felner 2002) poussent cette frontière encore
plus loin, jusqu'aux instances à ~66 coups de la littérature.

Les trois ingrédients de la technique :

1. **l'abstraction** (garder un motif, jokeriser le reste) ;
2. **le parcours rétrograde** (BFS / 0-1 BFS depuis le but abstrait = distances
   optimales, admissibles parce que le problème abstrait est un relâchement) ;
3. **la combinaison admissible additive** (somme sur des groupes disjoints, en ne
   comptant que les mouvements propres à chaque groupe).

La leçon plus large pour la Partie 3 : quand un problème est trop dur pour une
heuristique « instantanée », la bonne réponse n'est pas d'abandonner l'optimalité
mais de **précalculer une heuristique plus fine**, quitte à payer un coût de
construction amorti sur des millions de requêtes.

### Références

- **Korf, R. E. (1985).** *Depth-First Iterative-Deepening: An Optimal Admissible Tree Search.* AI 27(1).
- **Culberson, J. C. & Schaeffer, J. (1996).** *Searching with Pattern Databases.* AI\*AI.
- **Korf, R. E. & Felner, A. (2002).** *Disjoint Pattern Database Heuristics.* AI 134(1-2).
- **Felner, A., Korf, R. E., et al. (2004).** *Additive Pattern Database Heuristics.* JAIR 21.
- **Korf, R. E. (1997).** *Finding Optimal Solutions to Rubik's Cube Using Pattern Databases.* AAAI.
- **Edelkamp, S. & Schrödl, S. (2012).** *Heuristic Search: Theory and Applications.*
  Morgan Kaufmann. — Chapitre 8 (*Combining Heuristic Functions*) pose la
  notion de partage de coûts (cost partitioning) et discute la relaxation
  LP associée (cf. Felner et al. 2004, saturated cost partitioning). Ancre
  page mesurée sur l'index du PDF archivé : *action cost partitioning* /
  *pattern database partitioning*, **p. 312** (exemplaire tronqué : seules
  la p. 672 et l'index pp. 826-835 sont lisibles ; le numéro de chapitre,
  cohérent avec la TOC de l'édition, reste à confirmer sur un exemplaire
  complet).

***

← [Search-3 — Recherche informée](../Part1-Foundations/Search-03-Informed.ipynb)

| ↑ [Search (racine)](../README.md) | [Partie 4 — Métaheuristiques →](../Part4-Metaheuristics/README.md)
| 🐍 [Version Python](Search-03b-PatternDatabases.ipynb)

## À retenir

1. **Une PDB échange du calcul contre une table** : 524 160 entrées pour un groupe de 4 tuiles (P(16,4) = 43 680 motifs × 12 positions de la case vide), construites en 3,9 s ; 43 680 pour le groupe de 3. La table n'est pas une estimation : c'est l'optimum d'un problème plus facile, calculé une fois.
2. **L'admissibilité vient du relâchement, la somme de la partition de coûts** : chaque coup ne déplace qu'une tuile, donc ne coûte que dans une seule table — HAdditive = somme des quatre tables reste une borne inférieure, et HMax = 11 ≤ HAdditive = 28 sur l'instance de démonstration.
3. **La PDB additive repousse la frontière du résolvable** : sur I1 (opt 24) elle explore 732 nœuds contre 2 114 pour Manhattan (×2,9) ; sur I2 (opt 45) et I3 (opt 53), Manhattan dépasse 2 M de nœuds sans conclure quand la PDB additive résout à l'optimum (299 773 et 392 168 nœuds).
4. **« speedup NaN » est une réponse, pas un bug** : le facteur n'existe que si les deux heuristiques terminent ; quand Manhattan est hors budget, la comparaison se lit en « résolu / non résolu », pas en rapport de nœuds.
5. **La mécanique ne dépend pas du langage** : tables `byte[]`, encodage de rang injectif (19 975 états abstraits testés, 0 collision), construction des quatre tables en 9,2 s — la même structure qu'en Python, à la vitesse près.

## Vérifiez votre compréhension

1. Une table de groupe de 4 tuiles compte 524 160 entrées alors que P(16,4) vaut 43 680. D'où vient le facteur 12, et pourquoi la case vide doit-elle figurer dans l'état abstrait ?
2. Pourquoi HAdditive = somme des quatre tables reste-t-il admissible, alors qu'aucune table ne connaît la position des autres groupes ?
3. Sur I2 et I3, le tableau affiche « speedup NaN ». Que faut-il lire à la place du facteur, et pourquoi le carnet refuse-t-il d'inventer un nombre ?
4. L'encodage de rang est testé sur 19 975 états abstraits avec 0 collision. Quel défaut produirait une collision sur la table, et pourquoi le test est-il placé à la construction plutôt qu'à l'usage ?

<details><summary>Réponses</summary>

1. Un motif de 4 tuiles admet P(16,4) = 43 680 placements ; les coups applicables dépendent de la case vide, donc l'état abstrait est (placement du motif, case vide parmi les douze cases restantes) : × 12 = 524 160 — la cellule de lecture du carnet écrit exactement « 524 160 = P(16,4) × 12 ». Sans elle, le parcours rétrograde ne saurait pas quels mouvements existent.
2. Chaque coup déplace exactement une tuile, donc appartient à un seul groupe : le coût total se partage sans recouvrement entre les tables, chacune minore sa part, donc la somme minore le total — c'est la version C# du partage de coûts (section 6b), qui rend HMax ≤ HAdditive toujours vrai (la vérification des sorties rend `True`).
3. On lit « non résolu dans le budget » contre « résolu à l'optimum » : NaN signale que le dénominateur manque — Manhattan n'a pas conclu —, pas une mesure nulle ni une erreur. Inventer un facteur sur une heuristique qui n'a pas terminé serait un chiffre fabriqué ; le carnet préfère une case vide à un nombre faux.
4. Une collision ferait correspondre deux états abstraits distincts à la même entrée : l'un des deux lirait la distance de l'autre, et l'heuristique pourrait surestimer — un IDA* qui rend une solution sous-optimale sans le signaler. Testé à la construction, le défaut est attrapé avant tout usage en recherche (0 collision sur 19 975 états).

</details>

## Pour aller plus loin

- **Jumeau Python** : [Search-03b-PatternDatabases](Search-03b-PatternDatabases.ipynb) — la même technique en Python, avec ses propres mesures (×8,4 à ×20,6 sur ses trois instances) et ses sections sur les quotients (information mutuelle, dette Ruzsa).
- **Carnet voisin** : [Search-03c-LimitedDiscrepancySearch-CSharp](Search-03c-LimitedDiscrepancySearch-CSharp.ipynb) — borner l'exploration par la structure du problème plutôt que par une meilleure estimation.
- **Dans la même série** : [Search-03-Informed-CSharp](Search-03-Informed-CSharp.ipynb) — la dominance des heuristiques, dont HAdditive ≥ HMax est une instance mesurée ici.
- **Exemples guidés du carnet** : exemple guidé 1 (partition 6-6-3), exemple guidé 2 (PDB symétrique, Felner et al. 2004), exemple guidé 3 (généralisation au 24-puzzle) — résolutions créditées, section 12.
- **Exercices du carnet** : exercice 4 (partition 5-5-5), exercice 5 (max vs somme sur PDB non additives), exercice 6 (base de motifs sur les tours de Hanoï).
- **Lectures** : Korf & Felner (2002), *Disjoint Pattern Database Heuristics* — la partition 4-4-4-3 et la combinaison additive viennent de là.